In [ ]:
# Before starting this notebook, create a Google Earth Engine account and set up a cloud project
#Load Google Earth Engine and geemap Python libraries (you may have to install these before)

import ee
import geemap

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the project with your cloud project ID
ee.Initialize(project='Project ID')


In [ ]:
# load smap data from GEE and explore available bands 

smap_collection = (
    ee.ImageCollection('NASA/SMAP/SPL4SMGP/008')
    .filterDate('2020-01-01', '2020-01-10')
#    .filter(ee.Filter.calendarRange(6, 6, 'month'))
#    .first()
)
smap_collection

# Select one band - sm_rootzone
smap_bands = smap_collection.select(['sm_rootzone'])

# Apply Mean Reducer - calculates the mean of the sm_rootzone band
#smap_mean = smap_collection.mean()
mean_image_reducer = smap_bands.reduce(ee.Reducer.mean())

# Fix: Transform the geometry before clipping
#mean_image_reducer = image.reproject(
#    crs='EPSG:4326',
#    scale=1000
#)
#mean_image_reducer = mean_image_reducer2.transform(proj='EPSG:4326', maxError=1)


In [ ]:
# Load North America boundaries

states = ee.FeatureCollection("TIGER/2018/States")
lower48 = states.filter(ee.Filter.inList('NAME', [
    'Alabama', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 
    'Delaware', 'Florida', 'Georgia', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 
    'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 
    'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 
    'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 
   'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 
    'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 
    'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 
    'Wyoming'
]))

country_geometry = lower48.union()

In [ ]:
# Calculate zonal statistics and export to a .csv file (or your choice)

# Define output path 
out_stats = "/Users/amgi2571/Desktop/Spring 2026/remote sensing envi/final_proj/smap_zonal_stats.csv" 


def get_mean(raster, zones, output_file):
    """Input raster, zone (or study region), output file path, statistic type, and scale. Allowed output formats: csv, shp, json, kml, kmz. Allowed statistics type: MEAN, MAXIMUM, MINIMUM, MEDIAN, STD, MIN_MAX, VARIANCE, SUM """    
    zone_stats = geemap.zonal_stats(raster, zones, output_file, stat_type="MEAN", scale=1000)
    return print(zone_stats)

# Allowed output formats: csv, shp, json, kml, kmz 
# Allowed statistics type: MEAN, MAXIMUM, MINIMUM, MEDIAN, STD, MIN_MAX, VARIANCE, SUM 
#zone_stats = geemap.zonal_stats(smap_bands, country_geometry, out_stats, stat_type="MEAN", scale=1000)
#return zone_stats

In [ ]:
#calculate mean of sm_rootzone

get_mean(mean_image_reducer, country_geometry, out_stats)
#print(get_mean)

In [ ]:
# Create a visualization of sm_rootzone in the lower 48 states

m = geemap.Map(center=[40, -100], zoom=4)
clipped_image = mean_image_reducer.clip(country_geometry)

#select the SMAP band to visualize and color palette
vis_params = {
    'bands': ['sm_rootzone_mean'],
    'min': 0,
    'max': 0.9,
    'palette': 'Spectral',
}
m.add_layer(clipped_image, vis_params, 'Mean SM')
m

In [ ]:
# export image to HTML
#m.to_html(
#    filename="/Users/amgi2571/Desktop/Spring 2026/remote sensing envi/final_proj/mymap.html", title="Earth Engine Map", width='100%', height='800px'
#)

In [ ]:
#export to local computer
roi = ee.Geometry.Rectangle([-124.848974, 24.396308, -66.885444, 49.384358])
geemap.ee_export_image(clipped_image, filename="/Users/amgi2571/Desktop/Spring 2026/remote sensing envi/final_proj/test_sm_rootzone2.tif", scale=2000, region=roi)